# EDA Inicial - Customer Support Ticket Dataset

**Projeto de Bloco: Análise e Segurança de Agentes de IA - TP1**

# Documentação do Dataset

## 1. Fonte

O dataset é o **Customer Support Ticket Dataset**, publicado por *suraj520* no Kaggle em 02/06/2023, disponível em https://www.kaggle.com/datasets/suraj520/customer-support-ticket-dataset. Ele reúne cerca de 8,5 mil tickets de atendimento sobre produtos de tecnologia (celulares, notebooks, fones, eletrodomésticos inteligentes, etc.), com dados do cliente, do produto, do próprio ticket e de métricas de atendimento.

É um dataset **sintético**, ou seja, os registros foram gerados artificialmente e não vêm de um sistema real de atendimento. Isso aparece na inspeção dos dados, porque a descrição dos tickets vem com marcadores de template que nunca foram preenchidos e todos os e-mails usam domínios fictícios (`example.com`, `.net`, `.org`). A verificação de qualidade detalha esses pontos.

## 2. Principais características

* São **8.469 registros** e **17 colunas**, onde cada linha é uma solicitação de um cliente.
* As colunas cobrem quatro blocos: identificação do cliente (nome, e-mail, idade, gênero), o produto comprado e a data da compra, o ticket em si (tipo, assunto, descrição, status, prioridade, canal) e as métricas de atendimento (primeira resposta, tempo de resolução, resolução e nota de satisfação).
* Existem duas colunas que podem servir de alvo, `Ticket Type` (5 categorias) e `Ticket Subject` (16 categorias), as duas descrevendo o motivo do contato.
* Boa parte das métricas de atendimento só existe para tickets já fechados, então elas aparecem vazias na maioria dos registros por causa do ciclo de vida do ticket, não por erro.
* A coluna de descrição, que seria a "mensagem" do cliente, está preenchida com marcadores de template não substituídos, então ela não é utilizável como texto direto.

## 3. Motivo da escolha

Esse dataset encaixa no domínio de um sistema de atendimento ao cliente, porque cada linha é uma solicitação de suporte com o motivo do contato já rotulado. Isso permite tratar o problema como uma classificação de intenção. As perguntas que ele consegue responder são do tipo "qual é a intenção por trás desse contato?" e "como os tipos de solicitação se distribuem entre produtos, canais e perfis de cliente?".

Vale registrar uma limitação do dataset. Por ser sintético, ele funciona bem para praticar todo o fluxo de EDA e de construção da API, mas não representa um caso real, então serve mais como base de estudo do que como fonte para um modelo de produção.

# **Importação de Bibliotecas e Carregamento dos Dados**

In [ ]:
import pandas as pd
import numpy as np
import unicodedata

# No repositório, o CSV fica em data/
df = pd.read_csv("customer_support_tickets.csv")

def print_formatado(*args, prefixo=5, largura_total=100):
    titulo = " ".join(str(a) for a in args)
    print("\n")
    print("=" * 100)
    print(titulo)
    print("=" * 100)

# **Compreensão do Problema e do Dataset**

In [ ]:
print_formatado("Compreensão do problema e do dataset:")
print("Ver Markdown")

## **Observações Compreensão do Problema e do Dataset**

* **Domínio do problema:** classificar automaticamente a intenção de um cliente a partir dos dados do seu ticket de atendimento. É um problema de classificação com mais de duas classes.
* **Coluna-alvo:** `Ticket Type`, que já traz a intenção do contato em cinco categorias limpas (`Refund request`, `Technical issue`, `Cancellation request`, `Product inquiry`, `Billing inquiry`). A gente preferiu ela no lugar de `Ticket Subject` porque o Subject tem 16 categorias e funciona mais como tópico do que como intenção.
* **Variáveis candidatas a preditoras:** os atributos estruturados do ticket e do cliente, como produto comprado, canal, prioridade, idade e gênero.
* **O que seria uma solução:** um classificador que, recebendo os dados de um ticket, devolve a intenção mais provável, encaminhando o cliente para o fluxo de atendimento certo.
* **Ponto de atenção já nesta fase:** a descrição do ticket seria a fonte natural de texto, mas ela está preenchida com marcadores de template não substituídos, então não dá pra usar como mensagem. A gente detalha isso na verificação de qualidade.

# **Inspeção Inicial**

In [ ]:
print_formatado("1. Tamanho do dataset:")
print(df.shape)

In [ ]:
print_formatado("2. Variáveis do dataset:")
print(df.columns)

In [ ]:
print_formatado("3. Significado das colunas:")
print("Ver Markdown")

In [ ]:
print_formatado("4. Tipos dos dados:")
print(df.dtypes)

In [ ]:
print_formatado("5. Organização dos registros:")
df.info()
print("Ver Markdown")

In [ ]:
print_formatado("6. Exemplo de valores:")
print("\nPrimeiros registros:")
print(df.head())

print("\nÚltimos registros:")
print(df.tail())

print("\nRegistros aleatórios:")
print(df.sample(5, random_state=1))

print("\nExemplo dataset:")
display(df)

In [ ]:
print_formatado("7. Identificação das variáveis")

colunas_numericas = ["Customer Age", "Customer Satisfaction Rating"]
colunas_categoricas = ["Customer Gender", "Product Purchased", "Ticket Type",
                       "Ticket Subject", "Ticket Status", "Ticket Priority", "Ticket Channel"]
colunas_textuais = ["Customer Name", "Customer Email", "Ticket Description", "Resolution"]
colunas_temporais = ["Date of Purchase", "First Response Time", "Time to Resolution"]

print("\nVariáveis numéricas:")
for coluna in colunas_numericas:
    print(f"{coluna}: min={df[coluna].min()}, max={df[coluna].max()}, média={df[coluna].mean():.2f}")

print("\nVariáveis categóricas:")
for coluna in colunas_categoricas:
    print(f"{coluna} ({df[coluna].nunique()} categorias): {df[coluna].unique().tolist()}")

print("\nVariáveis textuais:")
for coluna in colunas_textuais:
    print(f"{coluna}: exemplo -> {str(df[coluna].dropna().iloc[0])[:80]}")

print("\nVariáveis temporais:")
for coluna in colunas_temporais:
    print(f"{coluna}: de {df[coluna].min()} até {df[coluna].max()}")

In [ ]:
print_formatado("8. Variáveis relevantes para o objetivo da análise")
print("Ver Markdown")

In [ ]:
print_formatado("9. Possíveis problemas nos dados")
print("Ver Markdown")

## **Observações Inspeção Inicial**

### 3. Significado das colunas

* **Ticket ID:** identificador único do ticket. Não é preditor, serve só de chave.
* **Customer Name:** nome do cliente. Texto livre, sem valor preditivo e com dado pessoal.
* **Customer Email:** e-mail do cliente. Todos usam domínios fictícios (`example.com`, `.net`, `.org`), coerente com dado sintético.
* **Customer Age:** idade do cliente, de 18 a 70 anos.
* **Customer Gender:** gênero do cliente (`Male`, `Female`, `Other`).
* **Product Purchased:** produto sobre o qual é o ticket, com 42 produtos de tecnologia.
* **Date of Purchase:** data da compra, entre 2020 e 2021.
* **Ticket Type:** tipo do ticket, em 5 categorias. É a nossa coluna-alvo.
* **Ticket Subject:** assunto do ticket, em 16 categorias. Segunda candidata a alvo.
* **Ticket Description:** descrição do problema. Deveria ser a mensagem do cliente, mas está com marcadores de template não preenchidos.
* **Ticket Status:** situação do ticket (`Open`, `Pending Customer Response`, `Closed`).
* **Resolution:** texto da resolução, preenchido só em tickets fechados.
* **Ticket Priority:** prioridade (`Low`, `Medium`, `High`, `Critical`).
* **Ticket Channel:** canal do contato (`Email`, `Phone`, `Chat`, `Social media`).
* **First Response Time:** momento da primeira resposta, vazio enquanto o ticket está aberto.
* **Time to Resolution:** momento da resolução, preenchido só em tickets fechados.
* **Customer Satisfaction Rating:** nota de 1 a 5 dada pelo cliente, só em tickets fechados.

### 5. Organização dos registros

* O dataset tem 8.469 registros e 17 colunas.
* O `df.info()` mostra colunas com contagem de não-nulos menor que o total, e isso vem das métricas de atendimento que só existem para tickets fechados, não de erro de carregamento.
* Os tipos foram lidos certos pelo pandas, com numéricas como `int64`/`float64`, e o resto como `object`. As colunas de data vieram como `object` e vão precisar virar datetime na limpeza.

### 6. Exemplo de valores

* Já dá pra ver nos exemplos que a `Ticket Description` está com o marcador `{product_purchased}` sem substituição, o que confirma o problema do texto.

### 7. Identificação das variáveis

* **Numéricas:** `Customer Age` e `Customer Satisfaction Rating`. O `Ticket ID` é inteiro, mas é identificador, então não conta como numérica de análise.
* **Categóricas:** `Customer Gender`, `Product Purchased`, `Ticket Type`, `Ticket Subject`, `Ticket Status`, `Ticket Priority` e `Ticket Channel`.
* **Textuais livres:** `Customer Name`, `Customer Email`, `Ticket Description` e `Resolution`.
* **Temporais:** `Date of Purchase`, `First Response Time` e `Time to Resolution`.

### 8. Variáveis relevantes para o objetivo da análise

* **Ticket Type** é o alvo.
* As candidatas a preditoras são os atributos estruturados do ticket e do cliente, como `Product Purchased`, `Ticket Channel`, `Ticket Priority`, `Customer Age` e `Customer Gender`.
* As métricas de atendimento (`Resolution`, `Time to Resolution`, `First Response Time`, `Customer Satisfaction Rating`) são resultados que só existem depois do atendimento, então não servem para prever a intenção no momento em que o ticket chega.
* `Ticket ID`, `Customer Name` e `Customer Email` são identificação e não entram como preditoras.

### 9. Possíveis problemas nos dados

* A `Ticket Description` está com marcadores de template não substituídos em todos os registros, então o texto não é utilizável direto.
* Várias colunas têm muitos valores vazios por causa do ciclo de vida do ticket, o que precisa ser interpretado como estado e não como erro.
* Há sinais de que o dataset é sintético, como e-mails com domínios fictícios e horários de primeira resposta e resolução concentrados num intervalo de dois dias de 2023. Isso não atrapalha o EDA, mas limita o dataset como base de um modelo real.

# **Verificação da Qualidade dos Dados**

In [ ]:
colunas_categoricas = ["Customer Gender", "Product Purchased", "Ticket Type",
                       "Ticket Subject", "Ticket Status", "Ticket Priority", "Ticket Channel"]

print_formatado("1. Valores ausentes:")
print("Valores ausentes (NaN) por coluna:")
print(df.isnull().sum())

print("\nValores ausentes (%):")
print((df.isnull().mean() * 100).round(2))

print("\nAusentes por Ticket Status (colunas de atendimento):")
for coluna in ["Resolution", "Time to Resolution", "Customer Satisfaction Rating", "First Response Time"]:
    print(f"\n{coluna}:")
    print(df.groupby("Ticket Status")[coluna].apply(lambda s: s.isnull().sum()))
print("Ver Markdown")

In [ ]:
print_formatado("2. Registros duplicados:")
print("Duplicatas completas:")
print(df.duplicated().sum())

print("\nDuplicatas desconsiderando Ticket ID:")
print(df.drop(columns=["Ticket ID"]).duplicated().sum())
print("Ver Markdown")

In [ ]:
print_formatado("3. Tipos de dados:")
print("Tipo predominante por coluna:")
for coluna in df:
    print(f"{coluna:<30}: {df[coluna].apply(type).value_counts().index[0]}")

print("\nQuantidade de categorias por coluna:")
for coluna in df:
    print(f"{coluna:<30}: {df[coluna].nunique()}")
print("Ver Markdown")

In [ ]:
print_formatado("4. Categorias escritas de maneira diferente:")
def normalizar_texto(valor):
    if pd.isna(valor):
        return valor
    valor = valor.strip().lower()
    valor = (
        unicodedata.normalize("NFKD", valor)
        .encode("ASCII", "ignore")
        .decode("utf-8")
    )
    return valor

for coluna in colunas_categoricas:
    com_normalizacao = df[coluna].apply(normalizar_texto).nunique()
    sem_normalizacao = df[coluna].nunique()
    print(f"{coluna:<20} -> Com Normalização: {com_normalizacao} | Sem Normalização: {sem_normalizacao}")

In [ ]:
print_formatado("5. Textos vazios ou espaços extras:")
for coluna in colunas_categoricas:
    qtd_espacos = (df[coluna] != df[coluna].str.strip()).sum()
    print(f"{coluna:<20}: {qtd_espacos}")

In [ ]:
print_formatado("6. Valores impossíveis ou fora do contexto:")
print("Customer Age:")
print(f"  mínimo: {df['Customer Age'].min()} | máximo: {df['Customer Age'].max()}")
print(f"  fora de 18 a 100: {df['Customer Age'][(df['Customer Age'] < 18) | (df['Customer Age'] > 100)].count()}")

print("\nCustomer Satisfaction Rating:")
print(f"  valores encontrados: {sorted(df['Customer Satisfaction Rating'].dropna().unique().tolist())}")
print(f"  fora de 1 a 5: {df['Customer Satisfaction Rating'][(df['Customer Satisfaction Rating'] < 1) | (df['Customer Satisfaction Rating'] > 5)].count()}")

print("\nDatas parseáveis (sem falha de conversão):")
for coluna in ["Date of Purchase", "First Response Time", "Time to Resolution"]:
    parsed = pd.to_datetime(df[coluna], errors="coerce")
    falhas = parsed.isna().sum() - df[coluna].isna().sum()
    print(f"  {coluna:<22}: falhas={falhas} | de {parsed.min()} até {parsed.max()}")

In [ ]:
print_formatado("7. Erros de digitação ou preenchimento:")

print("Marcadores de template não preenchidos na Ticket Description:")
placeholder = df["Ticket Description"].str.contains(r"\{.*?\}", regex=True, na=False)
print(f"  descrições com marcador {{...}}: {placeholder.sum()} ({placeholder.mean() * 100:.1f}%)")

print("")
print("Inconsistência de capitalização nas categóricas:")
for coluna in colunas_categoricas:
    if df[coluna].nunique() != df[coluna].str.lower().nunique():
        print(f"  {coluna}: há diferença de caixa")
    else:
        print(f"  {coluna}: sem inconsistência de capitalização")

print("")
print("Caracteres inesperados nas categóricas (fora de letras, espaço, hífen e '&'):")
for coluna in colunas_categoricas:
    padrao = r"[^A-Za-z\s\-&]"
    estranhos = df[coluna][df[coluna].str.contains(padrao, na=False, regex=True)].unique()
    if len(estranhos) > 0:
        print(f"  {coluna}: {list(estranhos)}")
    else:
        print(f"  {coluna}: nenhum caractere inesperado")

In [ ]:
print_formatado("8. Problemas que precisam ser corrigidos antes da análise:")
print("Ver Markdown")

## **Observações Qualidade dos Dados**

### 1. Valores ausentes e em qual quantidade
* Não há ausentes nas colunas de cliente, produto e nos rótulos (`Ticket Type` e `Ticket Subject` estão completos).
* As ausências se concentram nas métricas de atendimento, `Resolution` (67,3%), `Time to Resolution` (67,3%), `Customer Satisfaction Rating` (67,3%) e `First Response Time` (33,3%).
* Cruzando com `Ticket Status`, dá pra ver que essas ausências são **estruturais**. `Resolution`, `Time to Resolution` e `Customer Satisfaction Rating` só existem em tickets `Closed`, e `First Response Time` some só nos tickets `Open`. Ou seja, o vazio representa um estado do ticket, não dado perdido.
* **Decisão:** manter as ausências, porque preencher com "Desconhecido" ou remover as linhas destruiria informação e não faz sentido inventar uma resolução para um ticket que ainda não foi resolvido.

### 2. Registros duplicados
* Não há duplicatas completas nem duplicatas desconsiderando o `Ticket ID`, então nenhuma remoção é necessária.

### 3. Tipos dos dados
* As numéricas e o `Ticket ID` vieram corretas. As categóricas estão como `object` e vão virar `category`. As três colunas de data estão como `object` e precisam virar datetime.

### 4. Categorias escritas de forma diferente
* Nenhuma variável categórica muda de contagem depois da normalização, então não há grafias diferentes para a mesma categoria.

### 5. Textos vazios ou espaços extras
* Nenhuma categórica tem espaços sobrando nos valores.

### 6. Valores impossíveis ou fora de contexto
* `Customer Age` fica entre 18 e 70, dentro do esperado.
* `Customer Satisfaction Rating` só assume valores de 1 a 5, coerente com a escala.
* As três colunas de data são convertíveis sem falha. Vale notar que `First Response Time` e `Time to Resolution` caem todas num intervalo de dois dias de 2023, enquanto a compra é de 2020 e 2021, o que reforça que esses timestamps são artefato da geração sintética e não têm valor analítico.

### 7. Erros de digitação ou preenchimento
* O problema grave está na `Ticket Description`. Todos os registros contêm marcadores de template não substituídos, como `{product_purchased}`, e muitos ainda misturam fragmentos de código e texto sem sentido. O texto não serve como mensagem do cliente.
* Não há inconsistência de capitalização nem caractere inesperado nas categóricas.

### 8. Problemas que precisam ser corrigidos antes da análise
* Converter as categóricas para `category` e as datas para datetime.
* Manter as ausências estruturais das métricas de atendimento, documentando que representam o estado do ticket.
* Tratar a `Ticket Description` como não utilizável enquanto texto e não usá-la como preditora nesta etapa.
* Registrar que os timestamps de atendimento são artefato e não entram na análise.

# **Limpeza e Preparação dos Dados**

In [ ]:
print_formatado("1. Remover/tratar registros duplicados desnecessários:")
print(f"Duplicatas completas antes: {df.duplicated().sum()}")
df = df.drop_duplicates()
print(f"Duplicatas completas depois: {df.duplicated().sum()}")
print("Nenhuma duplicata a remover neste dataset.")

In [ ]:
print_formatado("2. Remover/tratar valores ausentes de forma justificada:")
print("Ausências por coluna (mantidas, pois são estruturais):")
colunas_atendimento = ["First Response Time", "Time to Resolution", "Resolution", "Customer Satisfaction Rating"]
for coluna in colunas_atendimento:
    print(f"{coluna:<30}: {df[coluna].isnull().sum()} ausentes")
print("\nDecisão: manter as ausências, pois representam o ciclo de vida do ticket (só existem quando o ticket avança/fecha).")
print("Ver Markdown")

In [ ]:
print_formatado("3. Corrigir tipos das colunas:")
colunas_categoricas = ["Customer Gender", "Product Purchased", "Ticket Type",
                       "Ticket Subject", "Ticket Status", "Ticket Priority", "Ticket Channel"]
colunas_data = ["Date of Purchase", "First Response Time", "Time to Resolution"]

print("Tipos antes da conversão:")
print(df[colunas_categoricas + colunas_data].dtypes)

for coluna in colunas_categoricas:
    df[coluna] = df[coluna].astype("category")
for coluna in colunas_data:
    df[coluna] = pd.to_datetime(df[coluna], errors="coerce")

print("\nTipos depois da conversão:")
print(df[colunas_categoricas + colunas_data].dtypes)

In [ ]:
print_formatado("4. Corrigir espaços, erros de preenchimento e inconsistências evidentes:")
print("Ver Markdown")

In [ ]:
print_formatado("5. Corrigir valores inválidos:")
print("Ver Markdown")

## **Observações Limpeza e Preparação dos Dados**

### 1. Remover/tratar registros duplicados
* Não havia duplicatas, então nenhuma linha foi removida.

### 2. Remover/tratar valores ausentes
* As ausências das métricas de atendimento foram mantidas de propósito, porque são estruturais e representam o estado do ticket. Preencher ou remover distorceria a base.
* Para a etapa de modelagem, essas colunas provavelmente vão ser descartadas como preditoras, já que são resultados posteriores ao contato e não estariam disponíveis no momento de prever a intenção. Isso fica registrado aqui, mas a decisão definitiva é da fase do modelo.

### 3. Corrigir tipos das colunas
* As sete colunas categóricas viraram `category` e as três colunas de data viraram datetime.

### 4. Corrigir espaços, erros de preenchimento e inconsistências evidentes
* Não há espaços extras nem inconsistências de grafia nas categóricas.
* A `Ticket Description` tem marcadores de template não substituídos, mas corrigir isso exigiria reconstruir o texto, o que está fora do escopo. A decisão é não usar essa coluna como texto nesta etapa e deixar a limitação documentada.

### 5. Corrigir valores inválidos
* Não há valores inválidos. Idade e nota de satisfação estão dentro das faixas esperadas e as datas convertem sem falha.